# Ensemble Models — Random Forest, Gradient Boosting & XGBoost
### Completed Notebook with Full Visualizations

In [ ]:
# Load the Wine dataset
# 178 wine samples from 3 cultivars (types of wine)
# 13 chemical measurement features: alcohol, malic acid, color intensity, etc.
# Goal: classify which cultivar a wine belongs to

from sklearn.datasets import load_wine
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print("Wine Dataset Overview")
print(f"  Samples: {wine.data.shape[0]}")
print(f"  Features: {wine.data.shape[1]}")
print(f"  Classes: {list(wine.target_names)}")
print()
df.head()

In [ ]:
# Split the data — standard 80/20 split
X = wine.data
y = wine.target

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

---
## Part 1: Random Forest (Bagging)

Random Forest = many Decision Trees, each trained on a **random subset** of the data, voting together.
- Each tree sees a different "bootstrap" sample of the training data
- Each split considers only a **random subset of features** (extra randomness!)
- Final prediction = majority vote across all trees

In [ ]:
# Train Random Forest
# n_estimators = number of trees
# max_depth = maximum depth per tree

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,   # 100 trees voting together
    max_depth=5,        # Each tree is constrained to depth 5
    random_state=42
)
rf.fit(X_train, y_train)
print("Random Forest trained with 100 trees!")

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score

y_pred_train = rf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"Train Accuracy: {train_accuracy:.4f} ({train_accuracy:.2%})")

In [ ]:
# Test accuracy
y_pred_test = rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")

In [ ]:
# Detailed classification report
# Precision: when model says class X, how often is it right?
# Recall: of all actual class X samples, how many did the model find?
# F1: harmonic mean of precision and recall

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_test, target_names=wine.target_names))

In [ ]:
# Feature Importance — which chemical measurements matter most?
importance = pd.Series(
    rf.feature_importances_,
    index=wine.feature_names
).sort_values(ascending=False)

print("Feature Importances (Random Forest):")
print(importance.round(4))

In [ ]:
# Visualization: Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
colors = ['#1565C0' if i == 0 else '#42A5F5' if i < 3 else '#90CAF9' 
          for i in range(len(importance))]
importance.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Feature Importance — Random Forest (Wine)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Importance Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=8)
for i, v in enumerate(importance):
    axes[0].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=7, fontweight='bold')

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=wine.target_names, yticklabels=wine.target_names)
axes[1].set_title('Confusion Matrix — Random Forest', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print(f"\nTop 3 most important features: {list(importance.index[:3])}")
print("Diagonal of confusion matrix = correct predictions!")

In [ ]:
# TUNING: How accuracy changes with number of trees
# This shows us the point of diminishing returns

test_scores = []
train_scores = []
n_trees_range = [1, 5, 10, 20, 25, 50, 100, 200, 500]

for n in n_trees_range:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train, y_train)
    train_scores.append(rf_temp.score(X_train, y_train))
    test_scores.append(rf_temp.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_trees_range, train_scores, 'o-', label='Train Accuracy', color='#2196F3', linewidth=2)
plt.plot(n_trees_range, test_scores,  's-', label='Test Accuracy',  color='#FF5722', linewidth=2)
plt.axvline(x=100, color='green', linestyle='--', alpha=0.7, label='n=100 (our choice)')
plt.fill_between(n_trees_range, train_scores, test_scores, alpha=0.1, color='red')
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs. Number of Trees in Random Forest', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.ylim(0.85, 1.02)
plt.tight_layout()
plt.show()

print("After ~50 trees, adding more gives very little benefit — diminishing returns!")

---
## Part 2: Gradient Boosting

Unlike Random Forest (parallel/independent trees), Gradient Boosting trains trees **sequentially**.
Each new tree learns to fix the **residual errors** of the previous trees.

Think of it as: Tree 1 makes a prediction → Tree 2 learns from Tree 1's mistakes → Tree 3 learns from Tree 2's mistakes...

In [ ]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,    # Number of trees (sequential)
    learning_rate=0.1,   # How much each tree contributes (smaller = more conservative)
    max_depth=3,         # Shallow trees work best for boosting
    random_state=42
)
gb.fit(X_train, y_train)

gb_train = accuracy_score(y_train, gb.predict(X_train))
gb_test  = accuracy_score(y_test,  gb.predict(X_test))

print("=== GRADIENT BOOSTING ===")
print(f"Train Accuracy: {gb_train:.2%}")
print(f"Test Accuracy:  {gb_test:.2%}")

In [ ]:
# Watch Gradient Boosting learn over iterations
# staged_predict_proba gives predictions after each boosting round

train_deviance = []
test_deviance  = []

for i, (y_train_pred, y_test_pred) in enumerate(
    zip(gb.staged_predict(X_train), gb.staged_predict(X_test))
):
    train_deviance.append(1 - accuracy_score(y_train, y_train_pred))
    test_deviance.append(1 - accuracy_score(y_test,   y_test_pred))

plt.figure(figsize=(10, 5))
iterations = range(1, len(train_deviance) + 1)
plt.plot(iterations, train_deviance, label='Train Error', color='#2196F3', linewidth=1.5)
plt.plot(iterations, test_deviance,  label='Test Error',  color='#FF5722', linewidth=1.5)
best_iter = np.argmin(test_deviance) + 1
plt.axvline(x=best_iter, color='green', linestyle='--', alpha=0.7, 
            label=f'Best iteration = {best_iter}')
plt.xlabel('Boosting Iteration (Tree Number)', fontsize=12)
plt.ylabel('Error Rate', fontsize=12)
plt.title('Gradient Boosting: Learning Curve Over Iterations', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best number of iterations: {best_iter} (minimum test error)")
print("Both errors decrease — model is learning, not memorizing!")

---
## Part 3: XGBoost — eXtreme Gradient Boosting

XGBoost is an optimized, faster version of Gradient Boosting with extra regularization.
It's one of the most popular models in ML competitions and industry.

In [ ]:
# XGBoost — eXtreme Gradient Boosting
# Faster, more regularized, and often more accurate than standard GradientBoosting
# Used widely in competitions (Kaggle) and industry

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    print("XGBoost not installed. Install with: pip install xgboost")
    xgb_available = False

if xgb_available:
    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        verbosity=0
    )
    xgb.fit(X_train, y_train)

    xgb_train = accuracy_score(y_train, xgb.predict(X_train))
    xgb_test  = accuracy_score(y_test,  xgb.predict(X_test))

    print("=== XGBoost ===")
    print(f"Train Accuracy: {xgb_train:.2%}")
    print(f"Test Accuracy:  {xgb_test:.2%}")

In [ ]:
# FINAL COMPARISON: All ensemble models side by side

from sklearn.tree import DecisionTreeClassifier

# Single Decision Tree as baseline
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

results = {
    'Decision Tree (depth=5)':   (accuracy_score(y_train, dt.predict(X_train)),
                                   accuracy_score(y_test,  dt.predict(X_test))),
    'Random Forest (n=100)':     (train_accuracy, test_accuracy),
    'Gradient Boosting':         (gb_train, gb_test),
}

if xgb_available:
    results['XGBoost'] = (xgb_train, xgb_test)

print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8}")
print("-" * 62)
for name, (train, test) in results.items():
    gap = train - test
    print(f"{name:<30} {train:>9.2%}  {test:>9.2%}  {gap:>7.2%}")

print("\nSmaller gap = less overfitting. Ensemble methods consistently beat single trees!")

In [ ]:
# VISUALIZATION: Model Comparison Bar Chart

model_names = list(results.keys())
train_accs  = [v[0] for v in results.values()]
test_accs   = [v[1] for v in results.values()]

x = np.arange(len(model_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar comparison
bars1 = axes[0].bar(x - width/2, train_accs, width, label='Train Acc', color='#2196F3', alpha=0.85)
bars2 = axes[0].bar(x + width/2, test_accs,  width, label='Test Acc',  color='#FF5722', alpha=0.85)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Train vs Test Accuracy by Model', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0.85, 1.05)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{bar.get_height():.2%}', ha='center', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{bar.get_height():.2%}', ha='center', fontsize=8)

# Overfitting gap
gaps = [t - v for t, v in zip(train_accs, test_accs)]
colors_gap = ['#4CAF50' if g < 0.05 else '#FF9800' if g < 0.10 else '#F44336' for g in gaps]
axes[1].bar(model_names, gaps, color=colors_gap, edgecolor='white')
axes[1].set_title('Overfitting Gap (Train − Test Accuracy)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Gap', fontsize=11)
axes[1].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
axes[1].axhline(y=0.05, color='orange', linestyle='--', alpha=0.7, label='5% threshold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
for i, g in enumerate(gaps):
    axes[1].text(i, g + 0.002, f'{g:.2%}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Ensemble Methods: Model Comparison on Wine Dataset',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Green bars = low overfitting | Orange = moderate | Red = high overfitting")

In [ ]:
# BONUS: Feature Importance Comparison — RF vs GradientBoosting vs XGBoost

fig, axes = plt.subplots(1, 3 if xgb_available else 2, figsize=(18, 5))

for ax, model, title in [
    (axes[0], rf, 'Random Forest'),
    (axes[1], gb, 'Gradient Boosting'),
] + ([(axes[2], xgb, 'XGBoost')] if xgb_available else []):
    imp = pd.Series(model.feature_importances_, index=wine.feature_names).sort_values(ascending=True)
    imp.plot(kind='barh', ax=ax, color='#42A5F5', edgecolor='white')
    ax.set_title(f'Feature Importance\n{title}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.axvline(x=0, color='black', linewidth=0.5)

plt.suptitle('Feature Importances Across Ensemble Methods',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Different algorithms may rank features differently — cross-check your findings!")